# 01 · Train the original models
### Machine Unlearning is Extinction, Not Forgetting

**What:** trains 3 ResNet-18 models (one per seed) on the **full** CIFAR-100 training set, forget classes
included.

**Why:** these are the models that "know" everything. Every unlearning method in notebook 03 starts from
them, and the extinction tests ask whether knowledge these models had comes back after unlearning.

**Outputs** (under *Output* when the run finishes):
- `ckpt/original_s{0,1,2}.pt`: the three models,
- `results/train_original__r*.jsonl`: accuracy of each model, including per forget class,
- `experiment_config.json`: the shared settings. Later notebooks check against this file so nobody
  accidentally changes, say, the width or the number of epochs halfway through.

**Time:** 3 models × ~10–15 min. With T4 x2 two models train at once, so about 25–35 minutes in total.

## Kaggle settings for this notebook

Set these in the right-hand **Settings** panel (in the editor: *View → Show sidebar* if it is hidden).

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **GPU T4 x2** | Two GPUs: `ext_utils` runs one independent worker per GPU, so two models train at the same time. Your weekly quota is counted in session hours, not per GPU, so T4 x2 roughly doubles what you get per hour. (A single P100 also works: the code detects one GPU and uses one worker.) |
| **Internet** | **On** | Needed to download CIFAR-100 (~170 MB) and, in notebook 04, CIFAR-100-C, unless you attach them as datasets. Internet requires a phone-verified Kaggle account. |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` between *interactive* sessions, so you don't lose files if the browser disconnects. Not needed for background runs. |
| **Environment** | *Always use latest* for the first run, then **Pin to original environment** | Pinning freezes the library versions (PyTorch etc.), so a Kaggle image update can't break a half-finished experiment. |
| **Language** | Python | |

**Inputs to attach** (*Add Input* → *Your Work* / *Your Datasets*):

| Input | Why |
|---|---|
| `ext-code` dataset (contains `ext_utils.py`) | The shared library. Upload it once as a private dataset. |
| **CIFAR-100 Python** dataset ([fedesoriano/cifar100](https://www.kaggle.com/datasets/fedesoriano/cifar100)) | **Strongly recommended.** Without it CIFAR-100 is downloaded from cs.toronto.edu at ~50 kB/s (56 min in run #1). Check that the setup cell prints `CIFAR-100: found at …`. |


**How to run.** Click **Save Version → Save & Run All (Commit)**. The notebook then runs in the background
for up to 12 hours with the browser closed. When it finishes, its files appear under *Output*, and the next
notebook attaches them as an input. 

**If a run stops early** (time budget or error), nothing is lost. Every finished job is written to disk
immediately. Attach this notebook's own latest output as an input and run it again: finished jobs are
skipped and their files are carried over.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Configuration

Every value below is the project default, written out explicitly so you can see and change it. The
settings marked *shared* must stay identical in all notebooks. Notebooks 02–06 verify this automatically
against `experiment_config.json`.

| Parameter | Value | What it does and why this value |
|---|---|---|
| `smoke` | `False` | `True` = a tiny dry run (30 training and 10 test images per class, 1 epoch, width 16) that finishes in minutes. Its files get a `_smoke` suffix, so they never mix with real results. Use it once to test the whole chain, then set `False`. **Must be the same in every notebook.** |
| `seeds` | `[0, 1, 2]` | Independent repetitions (different weight initialisation and data order). Three is the minimum for estimating variance. One statistical unit = (method, class, seed). |
| `forget_classes` | 10 classes | One class from each of 10 *different* CIFAR-100 superclasses, so no single superclass drives the result. The other 4 classes in each superclass are its "siblings" (semantic neighbours). |
| `norm` | `"bn"` | Normalization layer. `"bn"` = BatchNorm, the standard ResNet-18 of the unlearning literature (main experiments). `"gn"` = GroupNorm, a **control arm without running statistics**. The "BatchNorm illusion" (Kalani et al. 2026) showed that recomputing BatchNorm statistics alone can undo apparent forgetting. If the extinction signs also appear with GroupNorm, they are not a BatchNorm artifact. GroupNorm files get a `_gn` suffix, so both arms can live side by side. **Must be the same in every notebook of one arm.** |
| `pilot_class` | `"lamp"` | Used only for hyper-parameter calibration and the go/no-go pilot, then **excluded** from the main analysis. That way hyper-parameters are never tuned on the data that is later tested. |
| `width` | 64 | Channel width of ResNet-18 (64 = the standard model, 11.2 M parameters). |
| `epochs` | 40 | With the OneCycle schedule this reaches about 74–77 % test accuracy on CIFAR-100. The classic 200-epoch recipe gains only ~2–3 points for 5× the compute, which Kaggle's quota can't afford across ~34 models. |
| `batch_size` | 256 | Keeps a T4 busy with mixed precision. The learning rate below is tuned for this size. |
| `lr` | 0.1 | Peak learning rate of the OneCycle schedule. Standard for SGD on CIFAR at batch 256. |
| `pct_start` | 0.25 | Share of training spent warming the learning rate *up* before annealing it down. It stabilises the first epochs. |
| `momentum`, `nesterov` | 0.9, `True` | Standard SGD settings for ResNets. Nesterov momentum converges slightly faster. |
| `weight_decay` | 5e-4 | Standard L2 regularisation for ResNet-18 on CIFAR. |
| `label_smoothing` | 0.0 | Smoothing changes how confident the model is, which would distort the forgetting and recovery measurements. Plain cross-entropy keeps the models comparable with the literature. |
| `amp` | `True` | Mixed precision (fp16) uses the T4's tensor cores: about 2× faster and half the memory, with the same accuracy. |
| `channels_last` | `True` | A memory layout that makes convolutions faster on tensor-core GPUs. |
| `cudnn_benchmark` | `True` | cuDNN tries several convolution algorithms once and keeps the fastest (our image size never changes). It costs bit-exact reproducibility, which is acceptable because we repeat over 3 seeds. |
| `eval_batch_size` | 1024 | Evaluation needs no gradients, so bigger batches are safe and faster. |
| `ckpt_fp16` | `True` | Checkpoints are stored in half precision (≈22 MB instead of 45 MB) to stay well below Kaggle's 20 GB output limit. Every metric is computed *after* the same rounding, so reported numbers match the saved files exactly. |
| `retain_eval_subset` | 5000 | Number of training images used to estimate retain-*train* accuracy. All 49,500 would triple evaluation time for no real gain. |
| `time_budget_h` | 11.0 | Kaggle stops sessions at 12 h. The runner stops *starting* new jobs after ~11 h (it estimates each job's length), so the notebook finishes cleanly and its output is saved. Re-run to continue. |
| `n_workers` | `None` | `None` = one worker process per visible GPU (2 on T4 x2, 1 on P100). Set 1 to force a single worker, e.g. to see tracebacks directly. |
| `reuse_ckpt` | `True` | If a job's checkpoint already exists (e.g. saved just before a crash), load it instead of recomputing it. |
| `raise_on_error` | `True` | After all jobs have run, the cell raises an error if any job failed, so failures can't go unnoticed. Finished jobs are always kept. |
| `poll_seconds` | 60 | How often the progress line (`done/total`, elapsed minutes, last log line of each worker) is printed. |

In [ ]:
CFG = U.make_cfg(
    smoke=False,                    # True = quick dry run with tiny data (files get a _smoke suffix)
    seeds=[0, 1, 2],
    forget_classes=["maple_tree", "tiger", "rose", "shark", "bus",
                    "apple", "chair", "butterfly", "girl", "castle"],
    pilot_class="lamp",
    norm="bn",                      # "gn" = GroupNorm control arm (see notebook 01)
    width=64, epochs=40, batch_size=256, lr=0.1, pct_start=0.25,
    momentum=0.9, nesterov=True, weight_decay=5e-4, label_smoothing=0.0,
    amp=True, channels_last=True, cudnn_benchmark=True, eval_batch_size=1024,
    ckpt_fp16=True, retain_eval_subset=5000,
    time_budget_h=11.0, n_workers=None, reuse_ckpt=True, raise_on_error=True, poll_seconds=60,
)
print(U.save_experiment_config(CFG))

### Optional: the GroupNorm control arm

Once the main (BatchNorm) chain works, you can run a small second chain without BatchNorm. It answers the
reviewer question "is the recovery just the BatchNorm illusion?" directly. In **copies** of notebooks 01–06,
titled with `-gn` (e.g. `ext-01-originals-gn`), change only:

```python
norm="gn", seeds=[0], forget_classes=["maple_tree", "tiger", "rose", "shark", "bus"]
```

Everything else stays identical. Files get a `_gn` suffix, so they never mix with the main arm. With one seed
and five classes the arm costs about 4–6 GPU-hours. In notebook 04, the "BN frozen" and "BN recalibration"
variants are skipped automatically, because GroupNorm has no running statistics.

## Train

`U.jobs_train_original(CFG)` creates one job per seed. `U.launch(...)` then:

1. copies over any results and checkpoints from an earlier run of this notebook, if one is attached,
2. skips jobs that are already finished,
3. starts one worker process per GPU, which share the remaining jobs,
4. prints a progress line every `poll_seconds`.

Each worker keeps the whole dataset on its GPU as 8-bit integers and does the random-crop and flip
augmentation on the GPU. Kaggle has only 4 CPU cores, which would otherwise be the bottleneck.

Worker logs are written to `logs/train_original__w*.log`.

In [ ]:
jobs = U.jobs_train_original(CFG)
print(f"{len(jobs)} jobs:", [j["key"] for j in jobs])
U.launch("train_original", jobs, CFG)

## Check the results

For each seed you should see a **test accuracy of about 0.74–0.77** (lower in smoke mode, which is
expected). `forget_test_acc` is the model's accuracy on each forget class's 100 test images *before*
unlearning. That is the knowledge the unlearning methods must remove.

If a model is far below 0.70, something is wrong with training. Check the log before continuing.

In [ ]:
df = U.load_df("train_original", CFG)
print("test accuracy per seed:")
print(df.groupby("seed")["test_acc"].first().round(4).to_string())
low = df.groupby("seed")["test_acc"].first()
if not CFG["smoke"] and (low < 0.70).any():
    print("\nWARNING: a model is below 70% test accuracy - inspect logs/train_original__w*.log")
df.pivot_table(index="cls", columns="seed", values="forget_test_acc").round(3)

## Next step

1. Click **Save Version → Save & Run All (Commit)** if you ran this cell by cell.
2. Open **notebook 02** and attach this notebook's output (*Add Input → Your Work → 01 · Train the original
   models*).

## Name of this run's output

Title this notebook **`ext-01-originals`**. When the run has finished, create a dataset from its output with the
same name: open the notebook's viewer page → **Output** → **New Dataset** → title **`ext-01-originals`**, Private.
(Or skip the dataset and attach the notebook output directly: *Add Input → Your Work → Notebooks*.)

| Output name | Contains | Attached by |
|---|---|---|
| `ext-01-originals` | 3 original checkpoints, `train_original` results, `experiment_config.json` | notebooks 02, 03, 04, 05, 06 |

- **Re-runs and resumes:** don't create a new name. Add a **New Version** of the same dataset. Each output
  already contains everything from earlier runs (it's carried over), so only the latest version matters.
  In the notebooks that use it, update the input to the new version.
- **Smoke (dry) runs:** add `-smoke`, e.g. `ext-01-originals-smoke`, so real and test outputs never share a dataset.
- **GroupNorm control arm** (`norm="gn"`): add `-gn`, e.g. `ext-01-originals-gn`.